<a href="https://colab.research.google.com/github/disCarlosBustos/regresion_logistica/blob/main/1_regresion_logistica_fraude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Importar las librerías necesarias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
# URL del archivo en el bucket público de AWS
url = "https://hybridge-education-machine-learning-datasets.s3.us-east-1.amazonaws.com/Fraud.csv"
df = pd.read_csv(url)

In [3]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [4]:
len(df)

6362620

# ¡¡¡6,362,620 OBSERVACIONES!!!

In [5]:
# Distribución de la clase objetivo
print(df['isFraud'].value_counts())
print(df['isFraud'].value_counts(normalize=True) * 100)

isFraud
0    6354407
1       8213
Name: count, dtype: int64
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [6]:
print(df.groupby('type')['isFraud'].agg(['count','sum','mean']))
print(pd.crosstab(df['isFlaggedFraud'], df['isFraud']))

            count   sum      mean
type                             
CASH_IN   1399284     0  0.000000
CASH_OUT  2237500  4116  0.001840
DEBIT       41432     0  0.000000
PAYMENT   2151495     0  0.000000
TRANSFER   532909  4097  0.007688
isFraud               0     1
isFlaggedFraud               
0               6354407  8197
1                     0    16


In [10]:
# Preparacion: filtrar TRANSFER y CASH_OUT, quitar columnas y codificar type
df_f = df[df['type'].isin(['TRANSFER', 'CASH_OUT'])].copy()
df_f = df_f.drop(columns=['nameOrig', 'nameDest', 'isFlaggedFraud', 'step'])
df_f = pd.get_dummies(df_f, columns=['type'], drop_first=True, dtype=int)
print(df_f.shape, df_f['isFraud'].mean())

(2770409, 7) 0.002964544224336551


In [11]:
# Pasos 1-4: X/y, split 80/20 estratificado, escalado sin fuga, modelos A y B
# Prediccion previa: A tendra mejor precision y peor recall; B mejor recall y peor precision
X = df_f.drop(columns='isFraud')
y = df_f['isFraud']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
modelo_A = LogisticRegression(max_iter=1000).fit(X_train_s, y_train)
modelo_B = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train_s, y_train)
pred_A = modelo_A.predict(X_test_s)
pred_B = modelo_B.predict(X_test_s)
print('MODELO A (sin balanceo)')
print(confusion_matrix(y_test, pred_A))
print(classification_report(y_test, pred_A, digits=4))
print('MODELO B (class_weight=balanced)')
print(confusion_matrix(y_test, pred_B))
print(classification_report(y_test, pred_B, digits=4))

MODELO A (sin balanceo)
[[552340     99]
 [   929    714]]
              precision    recall  f1-score   support

           0     0.9983    0.9998    0.9991    552439
           1     0.8782    0.4346    0.5814      1643

    accuracy                         0.9981    554082
   macro avg     0.9383    0.7172    0.7903    554082
weighted avg     0.9980    0.9981    0.9978    554082

MODELO B (class_weight=balanced)
[[516641  35798]
 [   198   1445]]
              precision    recall  f1-score   support

           0     0.9996    0.9352    0.9663    552439
           1     0.0388    0.8795    0.0743      1643

    accuracy                         0.9350    554082
   macro avg     0.5192    0.9073    0.5203    554082
weighted avg     0.9968    0.9350    0.9637    554082



In [12]:
# Punto intermedio: distintos umbrales de decision para el modelo B
from sklearn.metrics import precision_score, recall_score
proba_B = modelo_B.predict_proba(X_test_s)[:, 1]
umbrales = [0.5, 0.7, 0.9, 0.95, 0.99, 0.995, 0.999]
tabla = pd.DataFrame([{'umbral': u, 'recall': recall_score(y_test, proba_B >= u), 'precision': precision_score(y_test, proba_B >= u), 'fraudes_detectados': int(((proba_B >= u) & (y_test == 1)).sum()), 'falsas_alarmas': int(((proba_B >= u) & (y_test == 0)).sum())} for u in umbrales])
print(tabla.round(4).to_string(index=False))

 umbral  recall  precision  fraudes_detectados  falsas_alarmas
  0.500  0.8795     0.0388                1445           35798
  0.700  0.8028     0.0708                1319           17316
  0.900  0.6969     0.1590                1145            6057
  0.950  0.6531     0.2332                1073            3528
  0.990  0.5666     0.4203                 931            1284
  0.995  0.5441     0.5156                 894             840
  0.999  0.4918     0.6865                 808             369


### Ajuste del umbral de decisión (modelo B)

| Umbral | Recall | Precisión | Fraudes detectados | Falsas alarmas |
|---|---|---|---|---|
| 0.50 | 0.88 | 0.04 | 1,445 | 35,798 |
| 0.70 | 0.80 | 0.07 | 1,319 | 17,316 |
| 0.90 | 0.70 | 0.16 | 1,145 | 6,057 |
| 0.95 | 0.65 | 0.23 | 1,073 | 3,528 |
| 0.99 | 0.57 | 0.42 | 931 | 1,284 |
| 0.999 | 0.49 | 0.69 | 808 | 369 |

Al subir el umbral, la precisión crece y el recall baja. Incluso con 0.999 el modelo B detecta más fraudes que el modelo A (808 contra 714), con más alarmas falsas (369 contra 99).

**Recomendación final: umbral de 0.90.** Conserva el 70 % de detección y reduce las falsas alarmas de 35,798 a 6,057 (un 83 % menos), de unas 25 a unas 5 por fraude real. Para un banco, ese punto mantiene bajo el costo de los fraudes que se escapan sin saturar al equipo de revisión ni bloquear a tantos clientes legítimos. Si la capacidad de revisión es menor, conviene subir a 0.95 o 0.99; el umbral debe fijarse con el costo real de cada error.

## Conclusiones

**Datos.** De 6,362,620 transacciones, solo 8,213 (0.13 %) son fraude. Con un desbalance así, la accuracy engaña: un modelo que nunca detecta fraude tendría 99.87 %. Por eso evaluamos con recall y precisión de la clase 1.

**Preprocesamiento.** Todo el fraude ocurre en TRANSFER y CASH_OUT, así que filtramos a esos dos tipos (2,770,409 filas; 0.30 % de fraude). Eliminamos `nameOrig` y `nameDest` (identificadores sin poder de generalización), `isFlaggedFraud` (el sistema anterior detectó solo 16 de 8,213 fraudes) y `step`. `type` quedó como una variable binaria con one-hot y `drop_first=True`. El escalador se ajustó solo con X_train para evitar fuga de datos.

**Resultados (prueba 20 %, 1,643 fraudes).**

| Modelo | Recall (fraude) | Precisión (fraude) | Fraudes no detectados | Falsas alarmas |
|---|---|---|---|---|
| A: sin balanceo | 0.43 | 0.88 | 929 | 99 |
| B: class_weight='balanced' | 0.88 | 0.04 | 198 | 35,798 |

La predicción previa se cumplió: A es más preciso y B detecta más fraude.

**Recomendación.** En un banco, un fraude que se escapa suele costar más que una alarma falsa, por lo que recomendamos el modelo B con un umbral de decisión de 0.90 (ver la sección de ajuste del umbral). Con el umbral por defecto de 0.5, B detecta el 88 % de los fraudes, pero genera unas 25 alarmas falsas por fraude real. Con 0.90 conserva el 70 % de detección (contra el 43 % del modelo A) y reduce las falsas alarmas de 35,798 a 6,057, unas 5 por fraude real. El umbral debe ajustarse según la capacidad de revisión del banco y el costo real de cada error. Como siguiente paso, crear variables de inconsistencia de saldos podría mejorar el modelo.

**Limitación.** El modelo solo aplica a transacciones TRANSFER y CASH_OUT; no debe usarse para otros tipos.

---

### Nota sobre el uso de IA

Este laboratorio se desarrolló con acompañamiento de un asistente de inteligencia artificial (Claude, de Anthropic), que, a petición del autor, escribió y ejecutó el código en este notebook y propuso las decisiones de preprocesamiento y la interpretación de resultados. El autor dirigió el análisis, revisó y aprobó cada decisión y es responsable de las conclusiones.

Eduardo Bustos